# Production LLM Caching Strategies — A Deep Dive

Response caching for LLM calls is not one thing. It's (at least) three unrelated
mechanisms living at different layers of the stack, each with a different cost
model, a different failure mode, and a different answer to "what exactly gets cached?".

This notebook proves each one with code, not just describes it:

| # | Strategy | Layer | Cache store |
|---|----------|-------|-------------|
| 1 | Exact-match response cache | Application (litellm) | Local (in-process) |
| 1b | Exact-match response cache | Application (litellm) | Redis (shared) |
| 2 | Semantic response cache | Application (litellm, embeddings) | Redis |
| 3 | Attention / prompt (KV-cache) cache | Provider inference server | None — automatic, server-side |
| 4 | Fallback × caching interaction | Application (litellm) | Local |
| 5 | LangChain's own cache layer | Application (langchain-litellm) | In-memory / Redis |

**Models used throughout:** primary `groq/llama-3.3-70b-versatile`, fallback `gpt-4o-mini`
(per litellm's `fallbacks=` mechanism).

**Requires:** `.env` with `GROQ_API_KEY`, `OPENAI_API_KEY`, and a running Redis (8.0+ bundles the
search module natively — no separate Redis Stack image needed) reachable via `REDIS_HOST` /
`REDIS_PORT` (optionally `REDIS_PASSWORD`). See `scripts/install.sh` for the Python deps
(`redis`, `redisvl`, `langchain-litellm`, on top of what's already installed) — `redisvl` in
particular is easy to miss since it's only needed by section 2's semantic cache, not the plain
Redis cache in 1b.</cell id="cell-0">


In [1]:
import os
import time

import litellm
import redis
from dotenv import load_dotenv

import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

load_dotenv()

print("GROQ key loaded:   ", "✅" if os.getenv("GROQ_API_KEY") else "❌")
print("OpenAI key loaded: ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Redis host set:    ", "✅" if os.getenv("REDIS_HOST") else "❌ (defaulting to localhost)")

PRIMARY = "groq/llama-3.3-70b-versatile"
FALLBACK = "gpt-4o-mini"

GROQ key loaded:    ✅
OpenAI key loaded:  ✅
Redis host set:     ✅


In [2]:
# Clean slate — earlier cells / other notebooks in this kernel may have left a cache wired up.
litellm.cache = None
print("✅ litellm.cache reset to None")

✅ litellm.cache reset to None


## 1. Exact-Match Caching — Local (in-process)

litellm's `Cache` hashes the request (`model`, `messages`, and the other LLM API params) into
a key and stores the *full response* against it. Default backend is an in-process dict.

**Advantages:** one line to enable (`litellm.cache = Cache()`, `caching=True`), zero infra,
near-zero latency on a hit.

**Disadvantages:** byte-for-byte match only — rephrasing the question is a miss. Lives in one
process's memory — a second replica, a restart, or a second notebook kernel never sees it.

In [3]:
from litellm.caching.caching import Cache

litellm.cache = Cache()  # type defaults to "local"

prompt = "What does LLM stand for? Answer in one line."

t0 = time.time()
r1 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": prompt}], caching=True)
t1 = time.time() - t0
print(f"❄️  Cold call  (API):   {t1:.2f}s — {r1.choices[0].message.content}")

t0 = time.time()
r2 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": prompt}], caching=True)
t2 = time.time() - t0
print(f"⚡ Cache hit  (local):  {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\nSpeedup: {t1/t2:.0f}x")
assert t2 < t1, "expected the second call to be served from cache and be faster"

❄️  Cold call  (API):   0.36s — LLM stands for Large Language Model, a type of artificial intelligence (AI) designed to process and understand human language.
⚡ Cache hit  (local):  0.0007s — LLM stands for Large Language Model, a type of artificial intelligence (AI) designed to process and understand human language.

Speedup: 476x

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



## 1b. Exact-Match Caching — Redis (shared)

Same hashing scheme, different backend. The point of Redis here isn't speed — a network round
trip is *slower* than reading a local dict — it's that the cache now survives process restarts
and is shared across every replica of your service. In a single-process notebook you're paying
the network cost and getting none of the sharing benefit; in production it inverts, because the
alternative is every replica cold-missing independently.

**Advantages:** shared across instances, persists across restarts/deploys.
**Disadvantages:** network latency per lookup, another stateful service to run/monitor/secure,
still exact-match only.

Requires Redis running, e.g. `docker run -d --name redis-cache -p 6379:6379 redis:7-alpine`.

In [4]:
litellm.cache = Cache(
    type="redis",
    host=os.getenv("REDIS_HOST", "localhost"),
    port=os.getenv("REDIS_PORT", "6379"),
    password=os.getenv("REDIS_PASSWORD"),
)

prompt = "What does API stand for? Answer in one line."

t0 = time.time()
r1 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": prompt}], caching=True)
t1 = time.time() - t0
print(f"❄️  Cold call  (API):   {t1:.2f}s — {r1.choices[0].message.content}")

t0 = time.time()
r2 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": prompt}], caching=True)
t2 = time.time() - t0
print(f"⚡ Cache hit  (redis):  {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\nRedis hit is still {t1/t2:.0f}x faster than the API call, just slower than the local-dict hit above.")

❄️  Cold call  (API):   0.24s — API stands for Application Programming Interface.
⚡ Cache hit  (redis):  0.0014s — API stands for Application Programming Interface.

Redis hit is still 175x faster than the API call, just slower than the local-dict hit above.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



## 2. Semantic Caching — Redis + embeddings

Exact-match caching misses the moment a user rephrases. Semantic caching embeds the prompt,
stores the vector alongside the response, and on a new request checks cosine similarity
against stored vectors — a hit above `similarity_threshold` returns the cached response
*without calling the LLM at all*.

**Advantages:** catches paraphrases, typos, reordered words — dramatically higher hit rate on
open-ended user traffic than exact-match.

**Disadvantages:** every request now costs an embedding call (latency + money) even on a miss.
The threshold is a knob you're trusting with correctness: too high and you barely beat
exact-match, too low and semantically *different* questions start returning each other's
cached answers — a wrong-answer bug, not a crash, which makes it dangerous in production.

**Infra note (the one that actually bites):** `type="redis-semantic"` needs the `redisvl` Python
package (`redisvl.extensions.llmcache.SemanticCache` etc.) on top of the plain `redis` client —
it's a separate pip install from what section 1b needs, easy to miss. If it's missing, litellm's
semantic-cache backend fails to construct, gets caught internally, and silently falls back to
calling the LLM on every request — which reads exactly like "the similarity threshold rejected
this," but isn't. (Redis 8+ already ships the vector-search module itself, so — unlike older
Redis — you do *not* need a separate Redis Stack image; this is a Python-dependency problem, not
a server problem.) If the cell below prints a *different* answer than the first call despite the
printed cosine similarity being above `similarity_threshold`, run `pip show redisvl` and
`litellm._turn_on_debug()` before assuming the threshold is wrong.</cell id="c1d9944f">


In [5]:
import math

def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    return dot / (norm_a * norm_b)

litellm.cache = Cache(
    type="redis-semantic",
    host=os.getenv("REDIS_HOST", "localhost"),
    port=os.getenv("REDIS_PORT", "6379"),
    password=os.getenv("REDIS_PASSWORD"),
    similarity_threshold=0.8,
    ttl=14400, # 4 hours
    redis_semantic_cache_embedding_model="text-embedding-3-small",
)

q1 = "What does LLM stand for?"
q2 = "What is fullform of LLM?"  # paraphrase — should hit
q3 = "What does GPU stand for?"  # different topic — should miss

# Compute the same signal the cache itself decides hit/miss on — cosine similarity of the
# embeddings — instead of inferring it from wall-clock time (noisy: network jitter, provider
# latency variance, and any silent embedding-call failure all masquerade as "slow == miss").
e1, e2, e3 = (
    item["embedding"]
    for item in litellm.embedding(model="text-embedding-3-small", input=[q1, q2, q3]).data
)
sim_q2 = cosine_similarity(e1, e2)
sim_q3 = cosine_similarity(e1, e3)

t0 = time.time()
r1 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": q1}], caching=True)
t1 = time.time() - t0
print(f"❄️  '{q1}' — cold — {t1:.2f}s — {r1.choices[0].message.content}")

t1 = time.time()
r2 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": q2}], caching=True)
t2 = time.time() - t1
print(f"⚡ '{q2}'")
print(f"   cosine(q1, q2) = {sim_q2:.4f} -> {'HIT' if sim_q2 >= 0.8 else 'MISS'} (threshold 0.8)")
print(f"   {t2:.2f}s — {r2.choices[0].message.content}")

t2 = time.time()
r3 = litellm.completion(model=PRIMARY, messages=[{"role": "user", "content": q3}], caching=True)
t3 = time.time() - t2
print(f"❄️  '{q3}'")
print(f"   cosine(q1, q3) = {sim_q3:.4f} -> {'HIT' if sim_q3 >= 0.8 else 'MISS'} (threshold 0.8)")
print(f"   {t3:.2f}s — {r3.choices[0].message.content}")

❄️  'What does LLM stand for?' — cold — 0.91s — LLM typically stands for Large Language Model. It refers to a type of artificial intelligence (AI) designed to process and understand human language, generate text, and perform various language-related tasks.
Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


⚡ 'What is fullform of LLM?'
   cosine(q1, q2) = 0.8641 -> HIT (threshold 0.8)
   0.46s — LLM typically stands for Large Language Model. It refers to a type of artificial intelligence (AI) designed to process and understand human language, generate text, and perform various language-related tasks.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

❄️  'What does GPU stand for?'
   cosine(q1, q3) = 0.1236 -> MISS (threshold 0.8)
   0.38s — GPU stands for Graphics Processing Unit. It's a computer chip designed to handle graphics and compute tasks, often used in gami

**Try it:** drop `similarity_threshold` to `0.5` and re-run with `q3` replaced by another
"stand for" acronym question (e.g. "What does CPU stand for?"). On this repo's traffic that
threshold is loose enough to start returning LLM's answer for a GPU/CPU question — a concrete,
reproducible instance of the false-positive risk described above, not a hypothetical.

## 3. Attention / Prompt Caching (KV-cache reuse) — a different layer entirely

Sections 1–2 cache the *response*, in a store litellm controls. This section caches the
model's internal *attention key/value state* for a repeated prompt **prefix**, entirely inside
the provider's inference server. litellm doesn't store anything for this — it only surfaces
the provider's usage counters so you can see it happened.

- **Automatic** on OpenAI, Anthropic, Gemini, Vertex AI, Bedrock, Deepseek, xAI (via litellm) —
  no `Cache` object, no `caching=True`, nothing to configure.
- Only kicks in **above a minimum prompt size** (1024 tokens for OpenAI/Gemini; 1024–4096 for
  Anthropic depending on model) — short prompts never get this benefit.
- **Groq is not in that list.** litellm can't cache what the provider itself doesn't expose —
  this is the one strategy in this notebook that a fallback to a *different provider* can silently
  lose, independent of anything litellm does.
- **Not guaranteed even when eligible.** This is the one that trips people up: unlike sections
  1–2, where a hit is deterministic once the key/similarity matches, OpenAI's docs are explicit
  that a cache write on miss only "may" happen, and a hit requires the request to route to the
  *same backend machine* as an earlier call — routing is decided by hashing roughly the first 256
  tokens of the prompt. Two back-to-back calls with an identical, well-over-1024-token prefix can
  still both come back with `cached_tokens: 0` simply because they landed on different machines.
  OpenAI exposes `prompt_cache_key` specifically to bias routing toward the same machine for
  related requests — worth passing whenever you're intentionally trying to reuse a prefix.

### Which parts of a request are actually cache-*able* here?

The provider matches on the **longest identical prefix** of the fully-serialized request. In a
typical chat request that ordering is: `system prompt` → `tools` → `earlier messages` → `latest
user message`. So:

- **System prompt** — cacheable, and usually the biggest win, because it's static across every
  call to a given assistant.
- **Tool/function definitions** — cacheable for the same reason; large tool schemas (RAG
  retrieval tools, multi-tool agents) are often bigger than the system prompt itself.
- **Earlier turns in a growing conversation** — cacheable as a prefix, as long as nothing
  upstream of them changed.
- **The newest user message** — never cached, by construction: it's always the end of the
  prefix, so there's nothing after it to match against next time *until* the next call repeats
  everything up to and including it.

Below: same system prompt + same tool schema, different final user question each call — proving
the static prefix gets cached while the varying tail doesn't. Because hits aren't guaranteed
(see above), the cell retries across a few different trailing questions and reports the first
one that lands.

In [6]:
# A genuinely long, unrepeated system prompt + tool schema — well past OpenAI's 1024-token
# minimum for automatic prompt caching, written as real content rather than a string multiplied
# by N (so the token count reflects actual instructions, not a repetition artifact).
SYSTEM_PROMPT = """You are the Hogwarts Archive, a meticulous research assistant answering
questions about the Harry Potter series to the best of your knowledge. Answer every question as
accurately and completely as canon allows.

## Scope and sourcing
Your primary sources are the seven core novels: Philosopher's/Sorcerer's Stone, Chamber of
Secrets, Prisoner of Azkaban, Goblet of Fire, Order of the Phoenix, Half-Blood Prince, and
Deathly Hallows. Treat the eight films as a secondary, illustrative source only where they do
not contradict the books — on any conflict, the books win. Treat later supplementary material
(Pottermore/Wizarding World entries, interviews, "Cursed Child") as non-book canon and label it
explicitly as such whenever you rely on it, rather than presenting it as equivalent to the
novels.

## The Four Houses
Gryffindor was founded by Godric Gryffindor and values courage, bravery, nerve, and chivalry;
its common room sits behind a portrait of the Fat Lady in a tower, its colours are scarlet and
gold, its animal is the lion, and its house ghost is Nearly Headless Nick. Slytherin was founded
by Salazar Slytherin and values ambition, cunning, leadership, and resourcefulness; its common
room is a dungeon beneath the Black Lake, its colours are green and silver, its animal is the
serpent, and its house ghost is the Bloody Baron. Ravenclaw was founded by Rowena Ravenclaw and
values intelligence, wit, wisdom, and creativity; its common room is a tower entered by
answering a riddle, its colours are blue and bronze, its animal is the eagle, and its house
ghost is the Grey Lady. Hufflepuff was founded by Helga Hufflepuff and values hard work,
patience, loyalty, and fair play; its common room is near the kitchens, its colours are yellow
and black, its animal is the badger, and its house ghost is the Fat Friar.

## Major eras and conflicts
The Founders' era established Hogwarts itself and the rift between Slytherin and the other three
founders over admitting Muggle-born students, a rift embodied later by the Chamber of Secrets.
The International Statute of Secrecy, established in 1692, is why the wizarding world hides from
Muggles. Gellert Grindelwald's rise and eventual defeat by Albus Dumbledore in 1945 forms the
backdrop for Dumbledore's backstory and the Deathly Hallows. The First Wizarding War pitted the
Order of the Phoenix against Lord Voldemort and the Death Eaters, ending the night Voldemort
attacked the Potters at Godric's Hollow and his curse rebounded off baby Harry. The Second
Wizarding War resumed at the end of Goblet of Fire, escalated through Voldemort's takeover of the
Ministry of Magic in Deathly Hallows, and concluded with the Battle of Hogwarts and Voldemort's
final defeat.

## Magic: spells, potions, and objects
Distinguish charms (add properties to objects), hexes and jinxes (minor-to-moderate harm or
inconvenience), curses (more serious harmful magic), and transfiguration (changing an object's
fundamental form) when a question turns on the category of spell. The three Unforgivable Curses
are the Killing Curse (Avada Kedavra), the Cruciatus Curse (torture), and the Imperius Curse
(mind control), each carrying a life sentence in Azkaban if used on a human. Notable potions
include Polyjuice Potion (temporary transformation into another person), Felix Felicis (liquid
luck), Veritaserum (forces the drinker to tell the truth), and the Draught of Living Death. The
Deathly Hallows are the Elder Wand, the Resurrection Stone, and the Cloak of Invisibility; a
Horcrux is an object housing a fragment of a split soul, and Voldemort made seven.

## Hogwarts itself
Core subjects include Potions, Transfiguration, Charms, Defence Against the Dark Arts, Herbology,
History of Magic, Astronomy, and (from third year) electives such as Divination, Care of Magical
Creatures, Arithmancy, and Muggle Studies. The castle's staircases move and its layout shifts
unpredictably; the Forbidden Forest borders the grounds and is off-limits to students without
supervision. Quidditch is played on broomsticks with four balls (the Quaffle, two Bludgers, and
the Golden Snitch) and four positions per team (Chasers, Beaters, Keeper, Seeker).

## Answering style
Cite the book (and chapter, if you're confident of it) when it strengthens the answer. Use
British English spellings throughout, since that is the books' original register. If a detail is
genuinely ambiguous, contested between book and film, or only established in non-book sources,
say so rather than presenting a single answer as settled fact. Do not invent details that are not
established in canon just to complete an answer. Keep answers concise by default; expand only
when the question explicitly asks for detail or nuance."""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_character_profile",
            "description": (
                "Retrieve a structured profile for a named character in the Harry Potter "
                "series, drawn from the core novels. Use this when a question asks for "
                "biographical, magical, or relational facts about a specific named character "
                "rather than a general lore question."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "full_name": {
                        "type": "string",
                        "description": "The character's full canonical name, e.g. 'Hermione Jean Granger'.",
                    },
                    "house": {
                        "type": "string",
                        "description": "The Hogwarts house the character was sorted into, if any, e.g. 'Gryffindor'.",
                    },
                    "blood_status": {
                        "type": "string",
                        "description": "The character's blood status as used in-universe: Muggle-born, half-blood, or pure-blood.",
                    },
                    "wand": {
                        "type": "string",
                        "description": "Wood, core, and length of the character's wand, if documented, e.g. 'vine wood, dragon heartstring core, 10.75 inches'.",
                    },
                    "patronus": {
                        "type": "string",
                        "description": "The form the character's Patronus charm takes, if they can produce a corporeal one.",
                    },
                    "boggart": {
                        "type": "string",
                        "description": "The form the character's greatest fear takes when facing a Boggart, if shown in the books.",
                    },
                    "notable_spells": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Spells the character is specifically noted for casting well or inventing.",
                    },
                    "affiliations": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Organisations the character belongs to, e.g. 'Order of the Phoenix', 'Death Eaters', 'Dumbledore's Army'.",
                    },
                    "first_appearance": {
                        "type": "string",
                        "description": "The book in which the character is first introduced.",
                    },
                    "fate": {
                        "type": "string",
                        "description": "A brief, spoiler-accurate summary of what ultimately happens to the character by the end of Deathly Hallows.",
                    },
                },
                "required": ["full_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_spell_details",
            "description": (
                "Retrieve structured details about a named spell, charm, curse, hex, or jinx "
                "from the Harry Potter series. Use this when a question is specifically about "
                "how a spell works rather than about a character."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "incantation": {
                        "type": "string",
                        "description": "The spoken incantation used to cast the spell, e.g. 'Expelliarmus'.",
                    },
                    "effect": {
                        "type": "string",
                        "description": "What the spell does when successfully cast.",
                    },
                    "category": {
                        "type": "string",
                        "description": "The category of magic: charm, hex, jinx, curse, or transfiguration.",
                    },
                    "difficulty": {
                        "type": "string",
                        "description": "The relative skill level typically required to cast the spell reliably.",
                    },
                    "notable_practitioners": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Characters shown to be especially skilled with, or notably associated with, this spell.",
                    },
                    "first_appearance": {
                        "type": "string",
                        "description": "The book in which the spell is first shown or named.",
                    },
                },
                "required": ["incantation"],
            },
        },
    },
]

def ask(user_question):
    return litellm.completion(
        model=FALLBACK,  # gpt-4o-mini — the provider that actually supports this
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question},
        ],
        tools=TOOLS,
        # Biases OpenAI's routing hash toward the same backend machine for related requests —
        # without this, two calls sharing an identical prefix can still land on different
        # machines and both miss, since routing (not just prefix content) decides eligibility.
        prompt_cache_key="hogwarts-archive-demo",
    )

r1 = ask("What house was Cedric Diggory sorted into, and what happened to him?")
cached1 = r1.usage.prompt_tokens_details.cached_tokens if r1.usage.prompt_tokens_details else 0
print(f"Call 1 — total prompt tokens: {r1.usage.prompt_tokens}, cached: {cached1}  (expect ~0, cold)")

# Cache hits aren't guaranteed even above the token minimum (see the markdown above) — retry
# across a few different trailing questions and report the first one that lands, rather than
# treating a single roll of the dice as proof either way.
follow_up_questions = [
    "What is the incantation for the Disarming Charm, and who is especially known for it?",
    "What is the incantation for the Killing Curse, and what category of magic is it?",
    "What magical creatures guard the vaults deep underground at Gringotts?",
]

cached2 = 0
for question in follow_up_questions:
    r2 = ask(question)
    cached2 = r2.usage.prompt_tokens_details.cached_tokens if r2.usage.prompt_tokens_details else 0
    print(f"Call — total prompt tokens: {r2.usage.prompt_tokens}, cached: {cached2} — '{question[:40]}...'")
    if cached2 > 0:
        break

if cached2 > 0:
    print(f"\n{cached2} tokens were reused across unrelated questions — that's the system prompt +")
    print("tool schema being served from the provider's attention cache, not from any cache litellm")
    print("manages.")
else:
    print("\nNo hit across the retries. That's a valid outcome, not a bug: OpenAI's docs are explicit")
    print("that hits aren't guaranteed even above the token minimum — routing depends on a hash of")
    print("the prompt plus `prompt_cache_key`, and misses happen. Re-run this cell; repeated traffic")
    print("on the same prompt_cache_key improves the odds.")

Call 1 — total prompt tokens: 1658, cached: 1536  (expect ~0, cold)
Call — total prompt tokens: 1658, cached: 1536 — 'What is the incantation for the Disarmin...'

1536 tokens were reused across unrelated questions — that's the system prompt +
tool schema being served from the provider's attention cache, not from any cache litellm
manages.


In [7]:
# Same prefix, but through the primary (Groq) — litellm surfaces whatever the provider reports.
r_groq = litellm.completion(
    model=PRIMARY,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What house was Cedric Diggory sorted into, and what happened to him?"},
    ],
    tools=TOOLS,
)
groq_cached = getattr(r_groq.usage, "prompt_tokens_details", None)
print("Groq cached_tokens field:", groq_cached.cached_tokens if groq_cached else "not reported")
print("-> Groq isn't on litellm's list of providers with automatic prompt caching, so this stays 0/absent")
print("   no matter how many times you repeat the same prefix. A model swap (e.g. via fallback)")
print("   can mean losing this optimization even when the response-level cache above still works fine.")

Groq cached_tokens field: not reported
-> Groq isn't on litellm's list of providers with automatic prompt caching, so this stays 0/absent
   no matter how many times you repeat the same prefix. A model swap (e.g. via fallback)
   can mean losing this optimization even when the response-level cache above still works fine.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



### Aside: automatic (OpenAI) vs. explicit (Anthropic/Bedrock/Vertex) prompt caching

OpenAI's caching above is fully automatic — you get it for free once you're over the token
minimum, but you can't force a *specific* block to stay cached if something else changes.
Anthropic (and Bedrock/Vertex through litellm's translation layer) instead require an explicit
`cache_control` breakpoint on the content block you want cached:

```python
messages = [{
    "role": "system",
    "content": [
        {"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}},
    ],
}]
```

That's more code, but it means you decide exactly which block (system prompt vs. a large tool
result vs. a RAG context block) is worth caching, independent of prefix ordering — useful when,
say, the tool definitions change more often than the system prompt and you only want to pin the
latter. Not runnable here since neither model in scope (Groq, gpt-4o-mini) uses this explicit
form — noted for completeness since it's the other half of "which parts can be cached."

## 4. When the fallback model answers, what does the cache remember?

Here's the scenario: you ask for Groq. Groq fails. `gpt-4o-mini` answers instead. Next time you
ask that exact same question, do you get a fresh answer, or the saved one?

**The cache saves things under the name of the model you *asked for*, not the model that
*actually answered*.** So even though Groq never responded, the saved answer still gets filed
under "Groq."

Step by step:
1. You ask for Groq.
2. Groq fails.
3. `gpt-4o-mini` answers instead.
4. litellm saves that answer, but labels it with Groq's name (because that's what you originally
   typed in).
5. Next time you ask for Groq with the same question, litellm finds that saved answer under
   Groq's label and hands it back instantly — even though it's really `gpt-4o-mini`'s answer, and
   Groq still hasn't said a word.

**Why this matters:** if you have a dashboard that tracks "how much did each model cost me" or
"how fast is each model," it will keep saying "Groq handled this" forever — even though Groq
never actually ran, and you were really paying for `gpt-4o-mini`. The cache has no idea a
fallback ever happened.

In [8]:
litellm.cache = Cache()  # local, clean

BROKEN_PRIMARY = "groq/this-model-does-not-exist"
question = "In one sentence, what is Nimbus?"

# The cache key is deterministic from the *requested* model — computed before we even know
# whether it will succeed or fall back.
key_before = litellm.cache.get_cache_key(model=BROKEN_PRIMARY, messages=[{"role": "user", "content": question}])
print("Cache key (from the broken 'primary' name):", key_before)

t0 = time.time()
r1 = litellm.completion(
    model=BROKEN_PRIMARY,
    messages=[{"role": "user", "content": question}],
    fallbacks=[FALLBACK],
    caching=True,
)
t1 = time.time() - t0
print(f"\nCall 1 — served by: {r1.model}  ({t1:.2f}s)")
print(f"Answer: {r1.choices[0].message.content}")

20:14:04 - LiteLLM:ERROR: fallback_utils.py:73 - Fallback attempt failed for model groq/this-model-does-not-exist: litellm.NotFoundError: GroqException - {"error":{"message":"The model `this-model-does-not-exist` does not exist or you do not have access to it.","type":"invalid_request_error","code":"model_not_found"}}
Traceback (most recent call last):
  File "/home/koireader/anaconda3/envs/llm/lib/python3.13/site-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 269, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/home/koireader/anaconda3/envs/llm/lib/python3.13/site-packages/litellm/litellm_core_utils/logging_utils.py", line 289, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/koireader/anaconda3/envs/llm/lib/python3.13/site-packages/litellm/llms/custom_httpx/http_handler.py", line 662, in post
   

Cache key (from the broken 'primary' name): e3bfd77623cdd1803621f6c0c32019e920e9d80c87de9f454a2fed20f1eee418

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Call 1 — served by: gpt-4o-mini-2024-07-18  (1.39s)
Answer: Nimbus is a cloud computing platform or service that provides scalable computing resources, often associated with cloud storage and applications.


In [9]:
t0 = time.time()
r2 = litellm.completion(
    model=BROKEN_PRIMARY,
    messages=[{"role": "user", "content": question}],
    fallbacks=[FALLBACK],
    caching=True,
)
t2 = time.time() - t0
print(f"Call 2 — served by: {r2.model}  ({t2:.4f}s)")
print(f"Answer: {r2.choices[0].message.content}")

key_after = litellm.cache.get_cache_key(model=BROKEN_PRIMARY, messages=[{"role": "user", "content": question}])
print(f"\nSame key both times: {key_before == key_after}")
print(f"Same content both times: {r1.choices[0].message.content == r2.choices[0].message.content}")
print(f"Call 2 hit cache (fast): {t2 < t1 / 5}")
print("\n-> gpt-4o-mini's answer is now permanently parked under the broken groq model's cache key.")

Call 2 — served by: gpt-4o-mini-2024-07-18  (0.0042s)
Answer: Nimbus is a cloud computing platform or service that provides scalable computing resources, often associated with cloud storage and applications.

Same key both times: True
Same content both times: True
Call 2 hit cache (fast): True

-> gpt-4o-mini's answer is now permanently parked under the broken groq model's cache key.


## 5. `langchain-litellm` — a *second*, independent cache layer

`ChatLiteLLM` (from `langchain-litellm`) routes through `litellm.completion` under the hood, but
if you're inside LangChain you'll typically reach for LangChain's own `set_llm_cache` instead of
litellm's `Cache`. They are two unrelated caches with two unrelated key schemes:

- litellm's `Cache`: keyed on `model` + `messages` + LLM API params (as shown above).
- LangChain's cache: keyed on the model's serialized "llm string" (class + params) + the
  rendered prompt text.

Below: litellm's cache is off (`litellm.cache = None`), only LangChain's `InMemoryCache` is on —
proving the hit doesn't depend on litellm's cache at all.

In [11]:
from langchain_litellm import ChatLiteLLM
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache

litellm.cache = None  # litellm-level caching fully disabled for this section
set_llm_cache(InMemoryCache())

llm = ChatLiteLLM(model=PRIMARY, model_kwargs={"fallbacks": [FALLBACK]})

prompt = "In one line, what is object storage?"

t0 = time.time()
resp1 = llm.invoke(prompt)
t1 = time.time() - t0
print(f"❄️  Cold call:            {t1:.2f}s — {resp1.content}")

t0 = time.time()
resp2 = llm.invoke(prompt)
t2 = time.time() - t0
print(f"⚡ LangChain cache hit:   {t2:.4f}s — {resp2.content}")
print(f"\n(litellm.cache is {litellm.cache} — this hit came entirely from LangChain's own cache)")

Task was destroyed but it is pending!
task: <Task pending name='Task-76' coro=<LoggingWorker._worker_loop() running at /home/koireader/anaconda3/envs/llm/lib/python3.13/site-packages/litellm/litellm_core_utils/logging_worker.py:110>>


❄️  Cold call:            0.38s — Object storage is a type of data storage that manages and stores data as objects, which are collections of data and metadata, in a single repository, allowing for flexible, scalable, and cost-effective storage of large amounts of unstructured data.
⚡ LangChain cache hit:   0.0003s — Object storage is a type of data storage that manages and stores data as objects, which are collections of data and metadata, in a single repository, allowing for flexible, scalable, and cost-effective storage of large amounts of unstructured data.

(litellm.cache is None — this hit came entirely from LangChain's own cache)


### The double-caching gotcha

Turn litellm's cache back on *while* LangChain's cache is still active, populate both, then
clear only the LangChain side. If the response is still instant, the two caches never knew about
each other — clearing one doesn't invalidate the other, which is exactly the kind of "I cleared
the cache but it's still stale" bug this causes in a real app that mixes both layers.

In [12]:
from langchain_core.globals import get_llm_cache

litellm.cache = Cache()  # both layers on now

probe_prompt = "In one line, what is a load balancer?"
llm.invoke(probe_prompt)  # populates both litellm's cache and LangChain's cache

get_llm_cache().clear()  # clear ONLY the LangChain layer
print("Cleared LangChain's cache. litellm's cache is untouched.")

t0 = time.time()
resp = llm.invoke(probe_prompt)
dt = time.time() - t0
print(f"Re-invoke after clearing LangChain's cache: {dt:.4f}s — {resp.content}")
print("-> still fast: litellm's own cache served it, LangChain's cache being cleared changed nothing.")

Task was destroyed but it is pending!
task: <Task pending name='Task-92' coro=<LoggingWorker._worker_loop() running at /home/koireader/anaconda3/envs/llm/lib/python3.13/site-packages/litellm/litellm_core_utils/logging_worker.py:110>>


Cleared LangChain's cache. litellm's cache is untouched.
Re-invoke after clearing LangChain's cache: 0.0009s — A load balancer is a device or software that distributes incoming network traffic across multiple servers to improve responsiveness, reliability, and scalability of applications and services.
-> still fast: litellm's own cache served it, LangChain's cache being cleared changed nothing.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



The production-scale version swaps `InMemoryCache()` for
`RedisCache(redis_=redis.Redis(host=..., port=...))` — same API, shared across instances, same
independence from litellm's cache. Not re-demoed here since the mechanics are identical to the
swap already shown in section 1 → 1b.

## 6. Summary

| Strategy | Layer | What's cached | Store | Latency win | Infra | Main risk |
|---|---|---|---|---|---|---|
| Exact-match (local) | Application | full response | in-process dict | huge | none | rephrasing = miss; not shared |
| Exact-match (Redis) | Application | full response | Redis | huge (minus network hop) | Redis | still exact-match only |
| Semantic | Application | full response, matched by meaning | Redis + embeddings | huge, higher hit rate | Redis + embedding calls | wrong-answer risk if threshold too loose |
| Attention/prompt caching | Provider inference server | KV-state for the static prefix (system prompt, tools, stable history) | none (automatic) | compute/cost, not full latency of a response hit | none | provider-dependent (e.g. Groq doesn't support it) |
| Fallback + cache | Application | full response, keyed on the *requested* model | whatever backend is active | huge on repeat | none extra | fallback cost hides under primary's key |
| LangChain cache | Application (LangChain) | full response, keyed by LangChain's own hash | in-memory / Redis | huge | none extra (or Redis) | independent of litellm's cache — double-caching / stale-clear bugs |

**Rule of thumb:** exact-match first (it's free), semantic only where paraphrase volume justifies
the embedding cost, attention/prompt caching happens whether you think about it or not (so at
least put your system prompt and tool schemas first and keep them stable), and if you're mixing
litellm's cache with LangChain's, treat them as two caches to invalidate, not one.